# Chapter 3 — Neural Networks for Language Modeling

## LLM Learning From Scratch

In this chapter, we will understand how a neural network learns to
predict the next token.

### Topics

- Neural networks
- Parameters and weights
- Forward propagation
- Activation functions
- Logits and probabilities
- Loss
- Cross-entropy
- Gradients
- Backpropagation
- Gradient descent
- Training loops

### Goal

Build a small neural language model using PyTorch and understand how
its parameters are updated during training.

2. What Is a Neural Network?
Cell 2 — Markdown
# 1. Neural Networks

A neural network is a mathematical function containing learnable
parameters.

It takes an input, performs a series of mathematical operations, and
produces an output.

Conceptually:

Input
  ↓
Layer
  ↓
Layer
  ↓
Output

During training, the network learns the values of its parameters so that
its predictions become better.

For language modeling:

Input:
previous tokens

Output:
prediction for the next token

The important idea:

A neural network is a function whose parameters are learned from data.

3. A Single Neuron
Cell 3 — Markdown

A simple neuron can be represented as:

x₁ ──►
       \
x₂ ───►  [ Weighted Sum ] ──► Activation ──► Output
       /
x₃ ──►

Mathematically:

$$ z = x_1w_1 + x_2w_2 + x_3w_3 + b $$

where:

x = inputs
w = weights
b = bias
z = weighted sum

Then an activation function can be applied.

In [1]:
import numpy as np
x=np.array([2.0,3.0,1.0])
w=np.array([0.5,-0.2,0.8])

b=0.1

z=np.dot(x,w)+b

print("Weighted sum:",z)

Weighted sum: 1.3


# 2. Activation Functions

Neural networks use activation functions to introduce non-linearity.

One common activation function is ReLU.

ReLU:

f(x) = max(0, x)

Therefore:

- negative values → 0
- positive values → unchanged

Example:

- ReLU(-2) = 0
- ReLU(3) = 3

In [2]:
def relu(x):
  return np.maximum(0,x)

values=np.array([2,-1,0,1,2,3])

print(relu(values))

[2 0 0 1 2 3]


A neural network combines many neurons into layers.

A simple network:

Input
  ↓
Linear Layer
  ↓
ReLU
  ↓
Linear Layer
  ↓
Output

Each linear layer contains learnable parameters.

For a language model, the final output represents scores for possible
next tokens.

In [4]:
import torch
import torch.nn as nn

model=nn.Sequential(
    nn.Linear(4,8),
    nn.ReLU(),
    nn.Linear(8,5)
)

print(model)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=5, bias=True)
)


# 3. Forward Propagation

Forward propagation is the process of passing an input through the
network to produce an output.

Example:

Input
  ↓
Layer 1
  ↓
Activation
  ↓
Layer 2
  ↓
Output

The output of the network is called its prediction.

For language modeling, the output consists of scores for possible
next tokens.

In [5]:
x=torch.tensor([1.0,2.0,0.5,-1.0])

output=model(x)



In [6]:
print("Output:",output)
print("Shape:",output.shape)

Output: tensor([-0.2311,  0.4543,  0.3992,  0.2391, -0.0750], grad_fn=<ViewBackward0>)
Shape: torch.Size([5])


# 4. Logits

The final layer of a language model produces a score for every token
in the vocabulary.

These raw scores are called **logits**.

For example:

Token      Logit

"I"        1.2
"am"       0.4
"happy"    3.1
"sad"      0.8
"today"    1.5

The largest logit corresponds to "happy".

However, logits are not probabilities.

We need to convert them into probabilities.

# 5. Softmax

Softmax converts logits into a probability distribution.

The probabilities:

- are between 0 and 1
- add up to 1

For logits:

[1.0, 2.0, 3.0]

softmax might produce:

[0.09, 0.24, 0.67]

The model therefore assigns the highest probability to the third token.

In [10]:
logits=torch.tensor([[1.0,2.0,3.0]])

probabilities=torch.softmax(logits,dim=1)

print(probabilities)
print("Sum:",probabilities.sum())

tensor([[0.0900, 0.2447, 0.6652]])
Sum: tensor(1.)


In [11]:
predicted_token=torch.argmax(probabilities,dim=1)

print("Predicted token ID:",predicted_token.item())

Predicted token ID: 2


# 6. Loss

Loss measures how far the model's prediction is from the correct answer.

Suppose the correct next token is:

"happy"

and the model predicts:

"I      → 0.10
am     → 0.10
happy  → 0.70
sad    → 0.05
today  → 0.05

The model is fairly confident in the correct answer.

The loss should therefore be relatively small.

If instead:

"happy" → 0.01

the model is very wrong.

The loss should be large.

The training objective is:

> Minimize the loss.

# 7. Cross-Entropy Loss

For classification problems such as next-token prediction, we commonly
use cross-entropy loss.

The basic idea is:

Correct token probability ↑
        ↓
Loss ↓

Correct token probability ↓
        ↓
Loss ↑

For one example:

Loss = -log(P(correct token))

Suppose:

P(correct token) = 0.8

Then:

Loss = -log(0.8)

This is relatively small.

If:

P(correct token) = 0.01

then the loss becomes much larger.

Therefore, cross-entropy encourages the model to assign high probability
to the correct next token.

In [12]:
logits=torch.tensor([[1.0,2.0,3.0]])
target=torch.tensor([2])

loss_fn=nn.CrossEntropyLoss()

loss=loss_fn(logits,target)

print("Loss:",loss.item())

Loss: 0.40760594606399536


# 8. Gradients

A gradient tells us how the loss changes when a parameter changes.

Suppose:

Loss = 2.0

and changing a particular weight slightly causes:

Loss ↓

That tells us that changing that weight in a particular direction is
helpful.

The gradient provides this direction mathematically.

For a parameter θ:

gradient = ∂Loss / ∂θ

The gradient tells us how sensitive the loss is to that parameter.

# 9. Backpropagation

Backpropagation calculates gradients for the parameters in the neural
network.

The basic process is:

1. Perform a forward pass.
2. Calculate the loss.
3. Calculate gradients by propagating information backward.
4. Use those gradients to update the parameters.

Conceptually:

Input
  ↓
Forward Pass
  ↓
Prediction
  ↓
Loss
  ↓
Backward Pass
  ↓
Gradients

In [16]:



model = nn.Linear(2, 1)

x = torch.tensor([[2.0, 3.0]])
target = torch.tensor([[1.0]])

output = model(x)

loss = ((output - target) ** 2).mean()

loss.backward()

print("Weight gradients:")
print(model.weight.grad)

print("\nBias gradient:")
print(model.bias.grad)

Weight gradients:
tensor([[ -8.3269, -12.4903]])

Bias gradient:
tensor([-4.1634])


# 10. Gradient Descent

Once we have gradients, we update the parameters.

The basic equation is:

θ_new = θ_old - learning_rate × gradient

where:

θ = parameter

The learning rate controls how large the update is.

Conceptually:

```text
Current parameters
       ↓
Calculate gradients
       ↓
Move parameters in the direction
that reduces the loss
       ↓
New parameters

In [19]:

model = nn.Linear(2, 1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

x = torch.tensor([[2.0, 3.0]])
target = torch.tensor([[1.0]])

output = model(x)

loss = ((output - target) ** 2).mean()

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Updated parameters:")
print(model.weight)
print(model.bias)

Updated parameters:
Parameter containing:
tensor([[0.0034, 0.4284]], requires_grad=True)
Parameter containing:
tensor([-0.6601], requires_grad=True)


# 11. Training Loop

Training a neural network means repeating the same process many times.

The basic training loop is:

```text
for each batch:

    1. Forward pass
    2. Calculate loss
    3. Clear old gradients
    4. Backpropagation
    5. Update parameters

# 12. Tiny Neural Language Model

We will create a small dataset:

"the cat sleeps"
"the cat eats"
"the dog sleeps"
"the dog eats"

Our task:

Given a word, predict the next word.

Example:

"the" → "cat"

"cat" → "sleeps"

"dog" → "eats"

This is a simplified language model.

Unlike Chapter 1, the model will now use a neural network and learn its
parameters through training.

In [20]:
text="""
the cat sleeps
the cat eats
the dog sleeps
the dog eats
"""

words=text.split()

vocabulary=sorted(set(words))

token_to_id={
    word:i
    for i,word in enumerate(vocabulary)
}

id_to_token={
    i:word
    for word,i in token_to_id.items()
}

print("Vocabulary:",vocabulary)
print("Token IDs:",token_to_id)

Vocabulary: ['cat', 'dog', 'eats', 'sleeps', 'the']
Token IDs: {'cat': 0, 'dog': 1, 'eats': 2, 'sleeps': 3, 'the': 4}


In [21]:
pairs=[]

for i in range(len(words)-1):
  current_word=words[i]
  next_word=words[i+1]

  pairs.append((
      token_to_id[current_word],
      token_to_id[next_word]
  ))

print(pairs)

[(4, 0), (0, 3), (3, 4), (4, 0), (0, 2), (2, 4), (4, 1), (1, 3), (3, 4), (4, 1), (1, 2)]


In [23]:
vocab_size=len(vocabulary)
embedding_dim=8
hidden_dim=16

model=nn.Sequential(
    nn.Embedding(vocab_size,embedding_dim),
    nn.Linear(embedding_dim,hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,vocab_size)
)

print(model)

Sequential(
  (0): Embedding(5, 8)
  (1): Linear(in_features=8, out_features=16, bias=True)
  (2): ReLU()
  (3): Linear(in_features=16, out_features=5, bias=True)
)


In [24]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

for epoch in range(1000):

    total_loss = 0

    for current_id, next_id in pairs:

        x = torch.tensor([current_id])
        target = torch.tensor([next_id])

        logits = model(x)

        loss = loss_fn(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch}, Loss: {total_loss:.4f}"
        )

Epoch 0, Loss: 17.4285
Epoch 100, Loss: 5.6895
Epoch 200, Loss: 5.6367
Epoch 300, Loss: 5.6114
Epoch 400, Loss: 5.5953
Epoch 500, Loss: 5.5853
Epoch 600, Loss: 5.5791
Epoch 700, Loss: 5.5749
Epoch 800, Loss: 5.5717
Epoch 900, Loss: 5.5697


In [25]:
def predict_next_word(word):

    token_id = token_to_id[word]

    x = torch.tensor([token_id])

    with torch.no_grad():
        logits = model(x)

    predicted_id = torch.argmax(logits, dim=-1).item()

    return id_to_token[predicted_id]


for word in vocabulary:
    print(
        f"{word} → {predict_next_word(word)}"
    )

cat → eats
dog → eats
eats → the
sleeps → the
the → dog
